# Private API Companion Notebook
This notebook holds the OpenAI vision workflow locally. Keep it untracked.


In [ ]:
# Step 5: Calling the ChatGPT API with Vision
# Objective: Send image and text to ChatGPT API and receive response.

import base64
import json
from io import BytesIO
from pathlib import Path

def encode_image(image_value):
    """Encode either a PIL image or a file path to base64."""
    if hasattr(image_value, "save"):
        buffer = BytesIO()
        image_value.save(buffer, format="JPEG")
        return base64.b64encode(buffer.getvalue()).decode("utf-8")

    with Path(image_value).open("rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

def prepare_request(image_value, prompt_text):
    """Prepare the request payload for the Responses API."""
    base64_image = encode_image(image_value)
    return {
        "model": "gpt-4.1",
        "input": [
            {
                "role": "user",
                "content": [
                    {"type": "input_text", "text": prompt_text},
                    {"type": "input_image", "image_url": f"data:image/jpeg;base64,{base64_image}"},
                ],
            }
        ],
    }

def call_chatgpt_api(image_value, prompt_text):
    request_payload = prepare_request(image_value, prompt_text)
    return openai_client.responses.create(**request_payload)

def handle_response(response):
    if response and getattr(response, "output_text", None):
        return response.output_text
    print("No response received")
    return None

def parse_json(output_text):
    if not output_text:
        return None
    try:
        parsed = json.loads(output_text)
        print("Parsed JSON:", parsed)
        return parsed
    except json.JSONDecodeError:
        print("Model output was not valid JSON:")
        print(output_text)
        return None

# Example usage with a dataframe image
image_value = products_df.iloc[0]["image"]
prompt_text = (
    "Analyze this product image and return JSON with keys: "
    "title, category, color, material, description."
)

response = call_chatgpt_api(image_value, prompt_text)
output_text = handle_response(response)
parse_json(output_text)

SyntaxError: invalid syntax. Perhaps you forgot a comma? (4250947698.py, line 17)

In [ ]:
# Step 6: Processing Multiple Products
# Objective: Generate listings for multiple products in batch.

def build_prompt(row):
    """Build a consistent prompt for each product."""
    return (
        "Analyze this product image and return JSON with keys: "
        "title, category, color, material, description. "
        f"Product context: {row.get('productDisplayName', '')}."
    )

def process_products(products_df, limit=5):
    """Generate listings for multiple products and collect results."""
    results = []

    for idx, row in products_df.head(limit).iterrows():
        try:
            image_value = row["image"]
            prompt_text = build_prompt(row)

            response = call_chatgpt_api(image_value, prompt_text)
            output_text = handle_response(response)
            parsed_output = parse_json(output_text)

            results.append({
                "index": idx,
                "productDisplayName": row.get("productDisplayName", ""),
                "listing": parsed_output if parsed_output is not None else output_text,
                "status": "success" if parsed_output is not None or output_text else "no_output",
            })

        except Exception as e:
            print(f"Error processing row {idx}: {e}")
            results.append({
                "index": idx,
                "productDisplayName": row.get("productDisplayName", ""),
                "listing": None,
                "status": "error",
                "error": str(e),
            })

    return pd.DataFrame(results)

# Example usage: process multiple products and save the results
batch_results = process_products(products_df, limit=5)
batch_results.to_csv("generated_listings.csv", index=False)
batch_results
